# Transformar APIs OpenAPI em ferramentas MCP usando o Bedrock AgentCore Gateway

## Visão Geral
O Bedrock AgentCore Gateway oferece aos clientes uma forma de transformar suas APIs existentes em servidores MCP totalmente gerenciados, sem a necessidade de gerenciar infraestrutura ou hospedagem. Os clientes podem trazer especificações OpenAPI em JSON ou YAML. Demonstraremos um agente de atendimento ao cliente usando APIs de suporte empresarial protegidas por OAuth2.

O fluxo de trabalho do Gateway envolve os seguintes passos para conectar seus agentes a ferramentas externas:
* **Criar as ferramentas para seu Gateway** - Defina suas ferramentas usando esquemas como especificações OpenAPI para APIs REST. As especificações OpenAPI são então analisadas pelo Amazon Bedrock AgentCore para criar o Gateway.
* **Criar um endpoint de Gateway** - Crie o gateway que servirá como ponto de entrada MCP com autenticação de entrada.
* **Adicionar alvos ao seu Gateway** - Configure os alvos OpenAPI que definem como o gateway roteia requisições para ferramentas específicas. Todas as APIs que fazem parte do arquivo OpenAPI se tornarão uma ferramenta compatível com MCP e serão disponibilizadas através da URL do seu endpoint Gateway. Configure a autorização de saída usando OAuth para cada alvo OpenAPI do Gateway. 
* **Atualizar o código do seu agente** - Conecte seu agente ao endpoint do Gateway para acessar todas as ferramentas configuradas através da interface MCP unificada.

![Como funciona](images/openapis-oauth-gateway.png)

### Detalhes do Tutorial


| Informação               | Detalhes                                                      |
|:-------------------------|:--------------------------------------------------------------|
| Tipo do tutorial         | Interativo                                                    |
| Componentes AgentCore    | AgentCore Gateway, AgentCore Identity                         |
| Framework de Agentes     | Strands Agent                                                 |
| Tipo de alvo do Gateway  | OpenAPI                                                       |
| Agente                   | Agente de suporte ao cliente                                  |
| IdP de Auth de Entrada   | Okta                                                          |
| Auth de Saída            | OAuth                                                         |
| Modelo LLM              | Anthropic Claude Haiku 4.5, Amazon Nova Pro                   |
| Componentes do tutorial  | Criação e Invocação do AgentCore Gateway                      |
| Vertical do tutorial     | Cross-vertical                                                |
| Complexidade do exemplo  | Fácil                                                         |
| SDK utilizado            | boto3                                                         |

Na primeira parte do tutorial, criaremos alguns alvos (targets) do AgentCore Gateway

### Arquitetura do Tutorial
Neste tutorial, transformaremos operações definidas em arquivo OpenAPI yaml/json em ferramentas MCP e as hospedaremos no Bedrock AgentCore Gateway.
Para fins de demonstração, construiremos um agente de suporte ao cliente que responde consultas relacionadas a tickets de suporte. O agente usa as APIs OpenAPI do Zendesk. A solução usa um Agente Langchain com modelos Amazon Bedrock.

## Pré-requisitos

Para executar este tutorial você precisará de:
* Jupyter notebook com Python 3.10+
* uv
* Credenciais AWS
* Okta
    - client_id
    - client_secret
    - Seu domínio Okta (ex.: dev-123456.okta.com)
    - Um ID de servidor de autorização OAuth2 (geralmente default)
* Zendesk integrado com Okta

In [ ]:
!pip install --force-reinstall -U -r requirements.txt --quiet

In [ ]:
# Set AWS credentials if not using SageMaker notebooks
import os
# os.environ['AWS_ACCESS_KEY_ID'] = '' # Set the access key
# os.environ['AWS_SECRET_ACCESS_KEY'] = '' # Set the secret key
os.environ['AWS_DEFAULT_REGION'] = os.environ.get('AWS_REGION', 'us-east-1')

In [ ]:
import os
import sys

# Get the directory of the current script
if '__file__' in globals():
    current_dir = os.path.dirname(os.path.abspath(__file__))
else:
    current_dir = os.getcwd()  # Fallback if __file__ is not defined (e.g., Jupyter)

# Navigate to the directory containing utils.py (one level up)
utils_dir = os.path.abspath(os.path.join(current_dir, '../..'))

# Add to sys.path
sys.path.insert(0, utils_dir)

# Now you can import utils
import utils

In [ ]:
#### Create an IAM role for the Gateway to assume
import utils

agentcore_gateway_iam_role = utils.create_agentcore_gateway_role("sample-lambdagateway")
print("Agentcore gateway role ARN: ", agentcore_gateway_iam_role['Role']['Arn'])

# Configurando o Okta para autorização de entrada no Gateway

Siga estes passos para criar um autorizador OAuth no Okta. 

* Se você não tem uma assinatura Okta, registre-se para um teste gratuito [aqui](https://www.okta.com/free-trial/).
* Faça login no console de administração do Okta 
* Siga as instruções [aqui](https://developer.okta.com/docs/guides/implement-grant-type/clientcreds/main/) para criar uma Aplicação com o tipo de concessão **Client Credentials**. 
* Após criar a aplicação, vá para a página de Aplicações e selecione a aplicação que acabou de criar. Salve o **Client ID** e o **Client Secret** em um editor de texto.
* Desabilite **Require Demonstrating Proof of Possession (DPoP)** em Configurações Gerais. 
* Abra a barra de navegação lateral e selecione Security -> API. Selecione o Servidor de Autorização Padrão que foi criado para você.
* Salve o valor de **Audience** em um editor de texto. 
* Salve o valor do **Issuer** (Deve ser semelhante a `https://trial-xxxxx.okta.com/oauth2/default`) em um editor de texto.
* Defina um Escopo Personalizado. Vá para a aba Scopes, clique em "Add Scope". Adicione um escopo chamado InvokeGateway. 
* Selecione **Access Policies**. Crie uma nova Política de Acesso. Dê um nome e Atribua a Todos os Clientes. Após a política ter sido criada, selecione **Add Rule**, deixe todos os valores como padrão e selecione **Create Rule**. 

# Criar o Gateway com autorizador Okta para autorização de entrada

In [ ]:
import boto3
from pprint import pprint
gateway_client = boto3.client('bedrock-agentcore-control', region_name = os.environ['AWS_DEFAULT_REGION'])

OKTA_DISCOVERY_URL="<Your Okta Issuer value>/.well-known/openid-configuration"
OKTA_AUDIENCE="<The audience value you saved earlier>" 

auth_config = {
        "customJWTAuthorizer": {
            "allowedAudience": [OKTA_AUDIENCE],
            "discoveryUrl": OKTA_DISCOVERY_URL
        }
}
create_response = gateway_client.create_gateway(name='OpenAPIOktaGwy2',
    roleArn = agentcore_gateway_iam_role['Role']['Arn'], # The IAM Role must have permissions to create/list/get/delete Gateway 
    protocolType='MCP',
    authorizerType='CUSTOM_JWT',
    authorizerConfiguration=auth_config, 
    description='AgentCore Gateway created from sdk with Okta Authorizer'
)
pprint(create_response)
# Retrieve the GatewayID used for GatewayTarget creation
gatewayID = create_response["gatewayId"]
gatewayURL = create_response["gatewayUrl"]

# Transformando APIs de suporte Zendesk em ferramentas MCP usando o Bedrock AgentCore Gateway

### Criar provedor de credenciais de autenticação de saída

In [ ]:
from botocore.config import Config
ZENDESK_DOMAIN="<Zendek domain url>"
ZENDESK_AUTH_ENDPOINT="https://<Zendeskl-domain>/oauth/authorizations/new"
ZENDESK_TOKEN_ENDPOINT="https://<Zendesk-domain>/oauth/tokens"
ZENDESK_CLIENT_ID="" # Your Zendesk OAuth client -  client id 
ZENDESK_SECRET=""  # Your Zendesk OAuth client -  client id 

sdk_config = Config(
    region_name=os.environ['AWS_DEFAULT_REGION'],
    retries={"max_attempts": 2, "mode": "standard"},
)

acps = boto3.client(
    service_name="bedrock-agentcore-control",
    config=sdk_config,
)

provider_config= {
    "customOauth2ProviderConfig": {
         "oauthDiscovery": {
             "authorizationServerMetadata": {
                 "issuer": ZENDESK_DOMAIN,
                 "authorizationEndpoint": ZENDESK_AUTH_ENDPOINT,
                 "tokenEndpoint": ZENDESK_TOKEN_ENDPOINT,
                 "responseTypes": ["token"]
             }
         },
         "clientId": ZENDESK_CLIENT_ID,
         "clientSecret": ZENDESK_SECRET
     }
 }

response = acps.create_oauth2_credential_provider(
    name="ZendeskOAuthTokenCfg", 
    credentialProviderVendor="CustomOauth2", 
    oauth2ProviderConfigInput=provider_config
)

pprint(response)
credentialProviderARN = response['credentialProviderArn']
pprint(f"Egress Credentials provider ARN, {credentialProviderARN}")

### Criar um alvo OpenAPI 

#### Fazer upload do arquivo OpenAPI yaml do Zendesk no S3

In [ ]:
# Create an S3 client
session = boto3.session.Session()
s3_client = session.client('s3')
sts_client = session.client('sts')

# Retrieve AWS account ID and region
account_id = sts_client.get_caller_identity()["Account"]
region = session.region_name
# Define parameters
bucket_name = '' # Your s3 bucket to upload the OpenAPI json file.
file_path = 'openapi-specs/Zendesk-support-apis.yaml'
object_key = 'Zendesk-support-apis.yaml'
# Upload the file using put_object and read response
try:
    with open(file_path, 'rb') as file_data:
        response = s3_client.put_object(Bucket=bucket_name, Key=object_key, Body=file_data)

    # Construct the ARN of the uploaded object with account ID and region
    openapi_s3_uri = f's3://{bucket_name}/{object_key}'
    print(f'Uploaded object S3 URI: {openapi_s3_uri}')
except Exception as e:
    print(f'Error uploading file: {e}')

#### Criar o alvo do gateway

Certifique-se de que a URL do servidor no arquivo OpenAPI está apontando para a URL do seu próprio endpoint. O Gateway lê a URL do servidor do arquivo OpenAPI e chama o endpoint. Antes de fazer o upload para o S3, certifique-se de fazer essa alteração.

In [ ]:
# S3 Uri for OpenAPI spec file
openapi_s3_target_config = {
    "mcp": {
          "openApiSchema": {
              "s3": {
                  "uri": openapi_s3_uri
              }
          }
      }
}

credential_config = [
    {
        "credentialProviderType" : "OAUTH",
        "credentialProvider": {
            "oauthCredentialProvider": {
                "providerArn": credentialProviderARN, 
                "scopes": ["tickets:read", "read", "tickets:write", "write"] 
            }
        }
    }
  ]

target_name="DemoOpenAPIGW"
response = gateway_client.create_gateway_target(
    gatewayIdentifier=gatewayID,
    name=target_name,
    description='OpenAPI Target with S3Uri using SDK',
    targetConfiguration=openapi_s3_target_config,
    credentialProviderConfigurations=credential_config)

# Printing the request ID and timestamp for you to report the defects. Please include them while reporting issues/defects  
response_metadata = response['ResponseMetadata']

# Chamando o Bedrock AgentCore Gateway a partir de um Agente Strands

O agente Strands integra-se perfeitamente com ferramentas AWS através do Bedrock AgentCore Gateway, que implementa a especificação do Model Context Protocol (MCP). Esta integração permite comunicação segura e padronizada entre agentes de IA e serviços AWS.

Em sua essência, o Bedrock AgentCore Gateway serve como um Gateway compatível com o protocolo que expõe APIs MCP fundamentais: ListTools e InvokeTools. Essas APIs permitem que qualquer cliente ou SDK compatível com MCP descubra e interaja com ferramentas disponíveis de forma segura e padronizada. Quando o agente Strands precisa acessar serviços AWS, ele se comunica com o Gateway usando esses endpoints padronizados pelo MCP.

A implementação do Gateway adere estritamente à (especificação de Autorização MCP)[https://modelcontextprotocol.org/specification/draft/basic/authorization], garantindo segurança robusta e controle de acesso. Isso significa que cada invocação de ferramenta pelo agente Strands passa por uma etapa de autorização, mantendo a segurança enquanto habilita funcionalidades poderosas.

Por exemplo, quando o agente Strands precisa acessar ferramentas MCP, ele primeiro chama ListTools para descobrir as ferramentas disponíveis e depois usa InvokeTools para executar ações específicas. O Gateway lida com todas as validações de segurança necessárias, traduções de protocolo e interações com serviços, tornando todo o processo transparente e seguro.

Essa abordagem arquitetural significa que qualquer cliente ou SDK que implemente a especificação MCP pode interagir com serviços AWS através do Gateway, tornando-o uma solução versátil e preparada para o futuro para integrações de agentes de IA.

# Solicitar o token de acesso do Okta para autorização de entrada

In [ ]:
print("Requesting the access token from Okta authorizer")
import requests
from requests.auth import HTTPBasicAuth

# Replace with your actual values
OKTA_DOMAIN = "Your Okta domain URL"
AUTH_SERVER_ID = "Okta app id"
CLIENT_ID = "<Okta client credentials client id>"
CLIENT_SECRET = "<Okta client credentials secret>"

TOKEN_URL = f"{OKTA_DOMAIN}/oauth2/{AUTH_SERVER_ID}/v1/token"

response = requests.post(
    TOKEN_URL,
    auth=HTTPBasicAuth(CLIENT_ID, CLIENT_SECRET),
    headers={"Content-Type": "application/x-www-form-urlencoded"},
    data={"grant_type": "client_credentials", "scope": "InvokeGateway"}
)

if response.status_code == 200:
    token = response.json()["access_token"]
    print("Access Token:", token)
else:
    print("Failed to get token:", response.status_code, response.text)

# Perguntar ao agente de suporte ao cliente com APIs Zendesk usando o Bedrock AgentCore Gateway

In [ ]:
from strands.models import BedrockModel
from mcp.client.streamable_http import streamablehttp_client 
from strands.tools.mcp.mcp_client import MCPClient
from strands import Agent

def create_streamable_http_transport():
    return streamablehttp_client(gatewayURL,headers={"Authorization": f"Bearer {token}"})

client = MCPClient(create_streamable_http_transport)

## The IAM group/user/ configured in ~/.aws/credentials should have access to Bedrock model
yourmodel = BedrockModel(
    model_id="us.amazon.nova-pro-v1:0",
    temperature=0.7,
)

In [ ]:
from strands import Agent
import logging


# Configure the root strands logger. Change it to DEBUG if you are debugging the issue.
logging.getLogger("strands").setLevel(logging.INFO)

# Add a handler to see the logs
logging.basicConfig(
    format="%(levelname)s | %(name)s | %(message)s", 
    handlers=[logging.StreamHandler()]
)

with client:
    # Call the listTools 
    tools = client.list_tools_sync()
    # Create an Agent with the model and tools
    agent = Agent(model=yourmodel,tools=tools) ## you can replace with any model you like
    #print(f"Tools loaded in the agent are {agent.tool_names}")
    #print(f"Tools configuration in the agent are {agent.tool_config}")
    
    # Invoke the agent with the sample prompt. This will only invoke  MCP listTools and retrieve the list of tools the LLM has access to. The below does not actually call any tool.
    #agent("Hi , can you list all tools available to you")
    agent("Count the number of support tickets")
    # Call the MCP tool explicitly. The MCP Tool name and arguments must match with your AWS Lambda function or the OpenAPI/Smithy API
    result = client.call_tool_sync(
    tool_use_id="count-tickets-1", # You can replace this with unique identifier. 
    name="DemoOpenAPIGW___CountTickets", # This is the tool name based on AWS Lambda target types. This will change based on the target name
    )
    #Print the MCP Tool response
    print(f"Tool Call result: {result['content'][0]['text']}")


# Limpeza
Recursos adicionais também são criados, como roles IAM, Políticas IAM, Provedor de credenciais, funções AWS Lambda, pools de usuários Cognito, buckets S3 que você pode precisar excluir manualmente como parte da limpeza. Isso depende do exemplo que você executar.

## Excluir o gateway (Opcional)

In [ ]:
import utils
utils.delete_gateway(gateway_client,gatewayID)